# 新思路 Alpha 构建实验

## 创新方向

### 1. 多时间尺度交叉验证
短期动量 vs 长期反转的组合

### 2. 非线性变换
sigmoid, tanh 平滑极值提高稳健性

### 3. 跨数据集融合
基本面 × 技术面, 情绪 × 期权

### 4. 动量反转组合
短期动量 + 长期反转背景

### 5. 市场状态自适应
高波动 → 反转, 低波动 → 动量

In [ ]:
from machine_lib import *

## Step 1: 登录

In [ ]:
s = login()

## Step 2: 获取数据字段

我们将使用多个数据集来构建跨维度的 alpha

In [ ]:
# 分析师数据集
analyst_df = get_datafields(s, dataset_id='analyst4', region='USA', universe='TOP3000', delay=1)
analyst_fields = process_datafields(analyst_df)
print(f"分析师字段数量: {len(analyst_fields)}")
print(analyst_fields[:3])

In [ ]:
# 基本面数据集
fundamental_df = get_datafields(s, dataset_id='fundamental6', region='USA', universe='TOP3000', delay=1)
fundamental_fields = process_datafields(fundamental_df)
print(f"基本面字段数量: {len(fundamental_fields)}")

## Step 3: 实验 1 - 多时间尺度交叉

In [ ]:
# 生成多时间尺度 alpha
mtf_alphas = multi_timeframe_factory(analyst_fields[:5], short_days=[5, 22], long_days=[66, 120])
print(f"多时间尺度 alpha 数量: {len(mtf_alphas)}")
print("样例:")
for alpha in mtf_alphas[:5]:
    print(f"  {alpha}")

In [ ]:
# 准备回测
init_decay = 6
mtf_alpha_list = [(alpha, init_decay) for alpha in mtf_alphas]
random.shuffle(mtf_alpha_list)

mtf_pools = load_task_pool_single(mtf_alpha_list, 3)
print(f"回测任务池数量: {len(mtf_pools)}")

In [ ]:
# 执行回测
single_simulate(mtf_pools, "SUBINDUSTRY", "USA", "TOP3000", 0)

## Step 4: 实验 2 - 非线性变换

In [ ]:
# 生成非线性变换 alpha
nl_alphas = nonlinear_transform_factory(analyst_fields[:5])
print(f"非线性变换 alpha 数量: {len(nl_alphas)}")
print("样例:")
for alpha in nl_alphas[:5]:
    print(f"  {alpha}")

In [ ]:
# 准备回测
nl_alpha_list = [(alpha, init_decay) for alpha in nl_alphas]
random.shuffle(nl_alpha_list)

nl_pools = load_task_pool_single(nl_alpha_list, 3)
single_simulate(nl_pools, "SUBINDUSTRY", "USA", "TOP3000", 0)

## Step 5: 实验 3 - 动量反转组合

In [ ]:
# 生成动量反转 alpha
mr_alphas = momentum_reversal_factory(analyst_fields[:10], momentum_window=22, reversal_window=120)
print(f"动量反转 alpha 数量: {len(mr_alphas)}")
print("样例:")
for alpha in mr_alphas[:5]:
    print(f"  {alpha}")

In [ ]:
mr_alpha_list = [(alpha, init_decay) for alpha in mr_alphas]
random.shuffle(mr_alpha_list)

mr_pools = load_task_pool_single(mr_alpha_list, 3)
single_simulate(mr_pools, "SUBINDUSTRY", "USA", "TOP3000", 0)

## Step 6: 实验 4 - 市场状态自适应

In [ ]:
# 生成状态自适应 alpha
ra_alphas = regime_adaptive_factory(analyst_fields[:8], threshold_days=66)
print(f"状态自适应 alpha 数量: {len(ra_alphas)}")
print("样例:")
for alpha in ra_alphas[:3]:
    print(f"  {alpha}")

In [ ]:
ra_alpha_list = [(alpha, init_decay) for alpha in ra_alphas]
random.shuffle(ra_alpha_list)

ra_pools = load_task_pool_single(ra_alpha_list, 3)
single_simulate(ra_pools, "SUBINDUSTRY", "USA", "TOP3000", 0)

## Step 7: 筛选有潜力的 Alpha

In [ ]:
# 获取所有实验的结果
# 注意: 需要根据实际回测日期调整
exp_tracker = get_alphas("08-01", "08-16", 1.0, 0.7, "USA", 100, "track")
print(f"有潜力的 alpha 数量: {len(exp_tracker)}")

In [ ]:
# 按数据字段前缀剪枝
exp_layer = prune(exp_tracker, 'anl4', 10)
print(f"剪枝后数量: {len(exp_layer)}")

# 查看表达式
for expr, decay in exp_layer[:5]:
    print(f"Decay={decay}: {expr}")

## Step 8: 二阶提升 - Group 操作

In [ ]:
so_alpha_list = []
group_ops = ["group_neutralize", "group_rank", "group_zscore"]

for expr, decay in exp_layer:
    for alpha in get_group_second_order_factory([expr], group_ops, "USA"):
        so_alpha_list.append((alpha, decay))

random.shuffle(so_alpha_list)
print(f"二阶 alpha 数量: {len(so_alpha_list)}")

In [ ]:
so_pools = load_task_pool_single(so_alpha_list, 3)
single_simulate(so_pools, 'SUBINDUSTRY', 'USA', 'TOP3000', 0)

## Step 9: 查看最佳结果

In [ ]:
# 获取高质量 alpha
final_tracker = get_alphas("08-01", "08-16", 1.3, 0.8, "USA", 200, "track")
print(f"高质量 alpha 数量: {len(final_tracker)}")

# 检查提交
stone_bag = [alpha[0] for alpha in final_tracker]
gold_bag = []
check_submission(stone_bag, gold_bag, 0)

print(f"\n可提交 alpha 数量: {len(gold_bag)}")
view_alphas(gold_bag)

## 总结

本 notebook 探索了 5 种创新的 alpha 构建思路:

1. **多时间尺度交叉**: 捕捉短期与长期动量的背离
2. **非线性变换**: 提高极值稳健性
3. **跨数据集融合**: 组合不同维度的信号
4. **动量反转组合**: 结合趋势与反转
5. **市场状态自适应**: 根据波动率调整策略

### 下一步

- 分析哪种思路产生了最多高质量 alpha
- 针对成功的方向深入扩展
- 尝试更复杂的组合 (例如: 三阶 trade_when)
- 测试不同的市场区域 (ASI, EUR, CHN)